## Check Notebook Kernel

Verify that the notebook is using the project virtual environment so package installs and imports come from the right Python.


In [95]:
import sys
print(sys.executable)

c:\Users\achur\desktop\AISE_CLASS_CLONES\march-madness-bracket-simulator\.venv\Scripts\python.exe


## Confirm Core Library Versions

Import the main analysis libraries and print their versions to confirm the environment is ready for data work.


In [96]:
import pandas as pd
import numpy as np
import sklearn

print(pd.__version__)
print(np.__version__)
print(sklearn.__version__)

2.3.3
2.4.3
1.8.0


## Inspect Raw Data Files

Point to the raw data folder and list available CSV files so we know what Kaggle data is present in the project.


In [97]:
from pathlib import Path

data_dir = Path("../data/raw")
sorted([p.name for p in data_dir.glob("*.csv")])[:20]


['Cities.csv',
 'Conferences.csv',
 'MConferenceTourneyGames.csv',
 'MGameCities.csv',
 'MMasseyOrdinals.csv',
 'MNCAATourneyCompactResults.csv',
 'MNCAATourneyDetailedResults.csv',
 'MNCAATourneySeedRoundSlots.csv',
 'MNCAATourneySeeds.csv',
 'MNCAATourneySlots.csv',
 'MRegularSeasonCompactResults.csv',
 'MRegularSeasonDetailedResults.csv',
 'MSeasons.csv',
 'MSecondaryTourneyCompactResults.csv',
 'MSecondaryTourneyTeams.csv',
 'MTeamCoaches.csv',
 'MTeamConferences.csv',
 'MTeamSpellings.csv',
 'MTeams.csv',
 'SampleSubmissionStage1.csv']

## Load Core Men's NCAA Datasets

This cell loads the main datasets needed for the project:
- teams
- regular season results
- tournament results
- seeds
- bracket slots


In [98]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data/raw")

m_teams = pd.read_csv(data_dir / "MTeams.csv")
m_reg = pd.read_csv(data_dir / "MRegularSeasonCompactResults.csv")
m_tourney = pd.read_csv(data_dir / "MNCAATourneyCompactResults.csv")
m_seeds = pd.read_csv(data_dir / "MNCAATourneySeeds.csv")
m_slots = pd.read_csv(data_dir / "MNCAATourneySlots.csv")




## Inspect Shapes and Columns

Print the size and column layout of each core dataset to understand the keys and what each file contains.


In [99]:
datasets = {
    "teams": m_teams,
    "regular_season": m_reg,
    "tourney": m_tourney,
    "seeds": m_seeds,
    "slots": m_slots,
}

for name, df in datasets.items():
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    display(df.head())

--- teams ---
shape: (381, 4)
columns: ['TeamID', 'TeamName', 'FirstD1Season', 'LastD1Season']


,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2026
1,1102,Air Force,1985,2026
2,1103,Akron,1985,2026
3,1104,Alabama,1985,2026
4,1105,Alabama A&M,2000,2026


--- regular_season ---
shape: (198079, 8)
columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT']


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,20,1228,81,1328,64,N,0
1,1985,25,1106,77,1354,70,H,0
2,1985,25,1112,63,1223,56,H,0
3,1985,25,1165,70,1432,54,H,0
4,1985,25,1192,86,1447,74,H,0


--- tourney ---
shape: (2585, 8)
columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT']


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,136,1116,63,1234,54,N,0
1,1985,136,1120,59,1345,58,N,0
2,1985,136,1207,68,1250,43,N,0
3,1985,136,1229,58,1425,55,N,0
4,1985,136,1242,49,1325,38,N,0


--- seeds ---
shape: (2626, 3)
columns: ['Season', 'Seed', 'TeamID']


,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374


--- slots ---
shape: (2586, 4)
columns: ['Season', 'Slot', 'StrongSeed', 'WeakSeed']


,Season,Slot,StrongSeed,WeakSeed
0,1985,R1W1,W01,W16
1,1985,R1W2,W02,W15
2,1985,R1W3,W03,W14
3,1985,R1W4,W04,W13
4,1985,R1W5,W05,W12


## Preview Bracket Slot Structure

Look at the first tournament slot rows to see how first-round seed matchups are encoded.


In [100]:
m_slots.head(15)


,Season,Slot,StrongSeed,WeakSeed
0,1985,R1W1,W01,W16
1,1985,R1W2,W02,W15
2,1985,R1W3,W03,W14
3,1985,R1W4,W04,W13
4,1985,R1W5,W05,W12
5,1985,R1W6,W06,W11
6,1985,R1W7,W07,W10
7,1985,R1W8,W08,W09
8,1985,R1X1,X01,X16
9,1985,R1X2,X02,X15


## Check Slot Season Coverage

Confirm the season range available in the tournament slot table.


In [101]:
m_slots["Season"].min(), m_slots["Season"].max()


(np.int64(1985), np.int64(2025))

## Inspect Latest Slot Layout

Preview the most recent season's slot rows to see how the current bracket structure is represented.


In [102]:
m_slots[m_slots["Season"] == m_slots["Season"].max()].head(20)


,Season,Slot,StrongSeed,WeakSeed
2519,2025,R1W1,W01,W16
2520,2025,R1W2,W02,W15
2521,2025,R1W3,W03,W14
2522,2025,R1W4,W04,W13
2523,2025,R1W5,W05,W12
2524,2025,R1W6,W06,W11
2525,2025,R1W7,W07,W10
2526,2025,R1W8,W08,W09
2527,2025,R1X1,X01,X16
2528,2025,R1X2,X02,X15


## Inspect Historical Bracket Progression

Use the 1985 slot rows to understand how later rounds depend on earlier slot winners.


In [103]:
m_slots[m_slots["Season"] == 1985].tail(20)


,Season,Slot,StrongSeed,WeakSeed
43,1985,R2Y4,R1Y4,R1Y5
44,1985,R2Z1,R1Z1,R1Z8
45,1985,R2Z2,R1Z2,R1Z7
46,1985,R2Z3,R1Z3,R1Z6
47,1985,R2Z4,R1Z4,R1Z5
48,1985,R3W1,R2W1,R2W4
49,1985,R3W2,R2W2,R2W3
50,1985,R3X1,R2X1,R2X4
51,1985,R3X2,R2X2,R2X3
52,1985,R3Y1,R2Y1,R2Y4


## Test Reusable Data Loaders

Load the same core datasets through the project's Python loader module to confirm reusable code works outside the notebook.


In [104]:
from march_madness_bracket_simulator.data_loader import load_core_march_madness_data

data = load_core_march_madness_data()

for name, df in data.items():
    print(name, df.shape)


teams (381, 4)
regular_season (198079, 8)
tourney (2585, 8)
seeds (2626, 3)
slots (2586, 4)


## Build First Team-Season Feature Table

Create a basic feature table from regular-season results with wins, losses, scoring, and margin statistics.


In [105]:
from march_madness_bracket_simulator.feature_engineering import build_team_season_features

team_features = build_team_season_features(data["regular_season"])
team_features.head()


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin
0,1985,1102,5.0,19.0,24.0,0.208333,63.083333,68.875000,-5.791667
1,1985,1103,9.0,14.0,23.0,0.391304,61.043478,64.086957,-3.043478
2,1985,1104,21.0,9.0,30.0,0.700000,68.500000,60.700000,7.800000
3,1985,1106,10.0,14.0,24.0,0.416667,71.625000,75.416667,-3.791667
4,1985,1108,19.0,6.0,25.0,0.760000,83.000000,75.040000,7.960000


## Merge Team Names and Seeds

Attach readable team names and tournament seeds to the team-season features so the table is easier to interpret.


In [106]:
team_features_named = team_features.merge(
    data["teams"][["TeamID", "TeamName"]],
    on="TeamID",
    how="left"
)

team_features_named = team_features_named.merge(
    data["seeds"],
    on=["Season", "TeamID"],
    how="left"
)

team_features_named.head()


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName,Seed
0,1985,1102,5.0,19.0,24.0,0.208333,63.083333,68.875000,-5.791667,Air Force,NaN
1,1985,1103,9.0,14.0,23.0,0.391304,61.043478,64.086957,-3.043478,Akron,NaN
2,1985,1104,21.0,9.0,30.0,0.700000,68.500000,60.700000,7.800000,Alabama,X07
3,1985,1106,10.0,14.0,24.0,0.416667,71.625000,75.416667,-3.791667,Alabama St,NaN
4,1985,1108,19.0,6.0,25.0,0.760000,83.000000,75.040000,7.960000,Alcorn St,NaN


## Review One Historical Season

Sort one older season by scoring margin to see whether the feature table surfaces strong teams sensibly.


In [107]:
team_features_named[team_features_named["Season"] == 1985].sort_values(
    "avg_scoring_margin", ascending=False
).head(15)


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName,Seed
80,1985,1207,25.0,2.0,27.0,0.925926,75.740741,60.074074,15.666667,Georgetown,W01
174,1985,1328,25.0,5.0,30.0,0.833333,89.833333,76.533333,13.300000,Oklahoma,Y01
118,1985,1256,26.0,2.0,28.0,0.928571,77.500000,64.821429,12.678571,Louisiana Tech,Y05
261,1985,1439,19.0,8.0,27.0,0.703704,81.148148,69.185185,11.962963,Virginia Tech,W09
217,1985,1385,27.0,3.0,30.0,0.900000,76.133333,64.400000,11.733333,St John's,X01
60,1985,1181,22.0,7.0,29.0,0.758621,79.241379,67.896552,11.344828,Duke,Y03
152,1985,1298,23.0,5.0,28.0,0.821429,78.214286,67.321429,10.892857,Navy,Z13
98,1985,1228,23.0,8.0,31.0,0.741935,68.225806,57.354839,10.870968,Illinois,W03
136,1985,1276,25.0,3.0,28.0,0.892857,79.750000,69.107143,10.642857,Michigan,Z01
103,1985,1234,20.0,10.0,30.0,0.666667,69.733333,59.266667,10.466667,Iowa,X08


## Step 1: Attach Team Names

Merge the feature table with the team lookup table using `TeamID` so rows are readable by school name.


In [108]:
team_features_named = team_features.merge(
    data["teams"][["TeamID", "TeamName"]],
    on="TeamID",
    how="left"
)

## Preview Features With Team Names

Check the feature table after adding team names to make sure the merge behaved correctly.


In [109]:
team_features_named.head()


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName
0,1985,1102,5.0,19.0,24.0,0.208333,63.083333,68.875000,-5.791667,Air Force
1,1985,1103,9.0,14.0,23.0,0.391304,61.043478,64.086957,-3.043478,Akron
2,1985,1104,21.0,9.0,30.0,0.700000,68.500000,60.700000,7.800000,Alabama
3,1985,1106,10.0,14.0,24.0,0.416667,71.625000,75.416667,-3.791667,Alabama St
4,1985,1108,19.0,6.0,25.0,0.760000,83.000000,75.040000,7.960000,Alcorn St


## Step 2: Attach Tournament Seeds

Merge tournament seeds onto the feature table using `Season` and `TeamID` because seeds change by season.


In [110]:
team_features_named = team_features_named.merge(
    data["seeds"],
    on=["Season", "TeamID"],
    how="left"
)

## Preview Tournament Teams Only

Filter to rows with a non-null seed to verify that tournament teams now carry bracket seed information.


In [111]:
team_features_named[team_features_named["Seed"].notna()].head(10)


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName,Seed
2,1985,1104,21.0,9.0,30.0,0.700000,68.500000,60.700000,7.800000,Alabama,X07
8,1985,1112,18.0,9.0,27.0,0.666667,66.518519,59.333333,7.185185,Arizona,X10
11,1985,1116,21.0,12.0,33.0,0.636364,65.333333,61.696970,3.636364,Arkansas,X09
14,1985,1120,18.0,11.0,29.0,0.620690,70.344828,66.655172,3.689655,Auburn,Z11
21,1985,1130,16.0,10.0,26.0,0.615385,74.884615,69.615385,5.269231,Boston College,Y11
53,1985,1173,19.0,9.0,28.0,0.678571,69.535714,64.607143,4.928571,Dayton,Z09
56,1985,1177,18.0,9.0,27.0,0.666667,71.666667,64.000000,7.666667,DePaul,W10
60,1985,1181,22.0,7.0,29.0,0.758621,79.241379,67.896552,11.344828,Duke,Y03
69,1985,1192,19.0,9.0,28.0,0.678571,67.892857,64.714286,3.178571,F Dickinson,Z16
80,1985,1207,25.0,2.0,27.0,0.925926,75.740741,60.074074,15.666667,Georgetown,W01


## Look at a Recent Season

Sort a recent season by scoring margin to get a more relevant feel for current-era team strength.


In [112]:
team_features_named[team_features_named["Season"] == 2025].sort_values(
    "avg_scoring_margin", ascending=False
).head(20)


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName,Seed
13098,2025,1181,31.0,3.0,34.0,0.911765,82.705882,61.911765,20.794118,Duke,W01
13128,2025,1211,25.0,8.0,33.0,0.757576,86.636364,69.636364,17.000000,Gonzaga,X08
13113,2025,1196,30.0,4.0,34.0,0.882353,85.411765,69.235294,16.176471,Florida,Z01
13137,2025,1222,30.0,4.0,34.0,0.882353,74.205882,58.470588,15.735294,Houston,X01
13378,2025,1471,28.0,4.0,32.0,0.875000,77.937500,62.843750,15.093750,UC San Diego,Y12
13183,2025,1268,25.0,8.0,33.0,0.757576,81.666667,67.000000,14.666667,Maryland,Z04
13041,2025,1120,28.0,5.0,33.0,0.848485,83.848485,69.606061,14.242424,Auburn,Y01
13342,2025,1433,27.0,6.0,33.0,0.818182,76.333333,62.515152,13.818182,VCU,W11
13313,2025,1403,25.0,8.0,33.0,0.757576,80.909091,67.575758,13.333333,Texas Tech,Z03
13295,2025,1385,30.0,4.0,34.0,0.882353,78.705882,65.882353,12.823529,St John's,Z02


## Look at Recent Seeded Teams

Focus on seeded teams in a recent season so the table is closer to the tournament use case.


In [113]:
team_features_named[
    (team_features_named["Season"] == 2025) &
    (team_features_named["Seed"].notna())
].sort_values("avg_scoring_margin", ascending=False).head(20)


,Season,TeamID,wins,losses,games,win_pct,avg_points_for,avg_points_against,avg_scoring_margin,TeamName,Seed
13098,2025,1181,31.0,3.0,34.0,0.911765,82.705882,61.911765,20.794118,Duke,W01
13128,2025,1211,25.0,8.0,33.0,0.757576,86.636364,69.636364,17.000000,Gonzaga,X08
13113,2025,1196,30.0,4.0,34.0,0.882353,85.411765,69.235294,16.176471,Florida,Z01
13137,2025,1222,30.0,4.0,34.0,0.882353,74.205882,58.470588,15.735294,Houston,X01
13378,2025,1471,28.0,4.0,32.0,0.875000,77.937500,62.843750,15.093750,UC San Diego,Y12
13183,2025,1268,25.0,8.0,33.0,0.757576,81.666667,67.000000,14.666667,Maryland,Z04
13041,2025,1120,28.0,5.0,33.0,0.848485,83.848485,69.606061,14.242424,Auburn,Y01
13342,2025,1433,27.0,6.0,33.0,0.818182,76.333333,62.515152,13.818182,VCU,W11
13313,2025,1403,25.0,8.0,33.0,0.757576,80.909091,67.575758,13.333333,Texas Tech,Z03
13295,2025,1385,30.0,4.0,34.0,0.882353,78.705882,65.882353,12.823529,St John's,Z02


## Load the 2026 Bracket File

Load the manually created bracket CSV so it can be matched against Kaggle team names and IDs.


In [114]:
bracket_2026 = pd.read_csv(data_dir / "bracket_2026.csv")
m_teams = pd.read_csv(data_dir / "MTeams.csv")

bracket_2026.head()

,region,seed,team
0,East,1,Duke
1,East,16,Siena
2,East,8,Ole Miss
3,East,9,TCU
4,East,5,St. John's


## First Pass Team Matching

Try a direct merge from bracket team names to Kaggle team names to see which teams map cleanly.


In [115]:
team_lookup = m_teams[["TeamID", "TeamName"]].copy()

merged = bracket_2026.merge(
    team_lookup,
    left_on="team",
    right_on="TeamName",
    how="left"
)

merged[["region", "seed", "team", "TeamID", "TeamName"]]


,region,seed,team,TeamID,TeamName
0,East,1,Duke,1181.0,Duke
1,East,16,Siena,1373.0,Siena
2,East,8,Ole Miss,NaN,NaN
3,East,9,TCU,1395.0,TCU
4,East,5,St. John's,NaN,NaN
...,...,...,...,...,...
59,Midwest,14,Wright St.,NaN,NaN
60,Midwest,7,Kentucky,1246.0,Kentucky
61,Midwest,10,Santa Clara,1365.0,Santa Clara
62,Midwest,2,Iowa St.,NaN,NaN


## Find Unmatched Bracket Teams

List bracket team names that failed to match so we can normalize them before modeling.


In [116]:
merged[merged["TeamID"].isna()][["region", "seed", "team"]]


,region,seed,team
2,East,8,Ole Miss
4,East,5,St. John's
10,East,3,Michigan St.
11,East,14,North Dakota St.
14,East,2,UConn
17,South,16,Lehigh / PVAMU
21,South,12,McNeese
28,South,7,Saint Mary's
33,West,16,Long Island
35,West,9,Utah St.


## Create a Manual Name Mapping

Define known name corrections where the bracket display names do not exactly match Kaggle team names.


In [117]:
name_map = {
    "Ole Miss": "Mississippi",
    "St. John's": "St John's",
    "Michigan St.": "Michigan St",
    "North Dakota St.": "North Dakota St",
    "UConn": "Connecticut",
    "McNeese": "McNeese St",
    "Saint Mary's": "Saint Mary's CA",
    "Long Island": "LIU Brooklyn",   # may need checking
    "Utah St.": "Utah St",
    "Kennesaw St.": "Kennesaw",
    "Miami (FL)": "Miami FL",
    "Queens (N.C.)": "Queens NC",
    "Saint Louis": "St Louis",
    "Wright St.": "Wright St",
    "Iowa St.": "Iowa St",
    "Tennessee St.": "Tennessee St",
}


## Apply Name Cleanup

Create a cleaned team-name column that will be used for a second, more accurate team-ID merge.


In [118]:
bracket_2026["team_clean"] = bracket_2026["team"].replace(name_map)


## Step 1: make a copy of the bracket table

In [119]:
bracket_clean = bracket_2026.copy()


### Why:

so you don’t keep mutating the original
it gives you a clean working version

## Step 2: add a play-in flag

In [120]:
bracket_clean["is_play_in"] = bracket_clean["team"].str.contains("/", regex=False)


### What this means:

if the team name has /, it is a play-in row
those are not normal one-team bracket rows yet

In [121]:
bracket_clean[["team", "is_play_in"]].head(20)


,team,is_play_in
0,Duke,False
1,Siena,False
2,Ole Miss,False
3,TCU,False
4,St. John's,False
5,Northern Iowa,False
6,Kansas,False
7,Cal Baptist,False
8,Louisville,False
9,South Florida,False


## Step 3: build the cleaned-name column

In [122]:
bracket_clean["team_clean"] = bracket_clean["team"].replace(name_map)


### Why:

- keep original team
- create a cleaned version for matching

In [123]:
bracket_clean[["team", "team_clean", "is_play_in"]].head(25)


,team,team_clean,is_play_in
0,Duke,Duke,False
1,Siena,Siena,False
2,Ole Miss,Mississippi,False
3,TCU,TCU,False
4,St. John's,St John's,False
5,Northern Iowa,Northern Iowa,False
6,Kansas,Kansas,False
7,Cal Baptist,Cal Baptist,False
8,Louisville,Louisville,False
9,South Florida,South Florida,False


## What I'm checking:

- normal names stay normal
- mismatched names change to Kaggle-style names
- play-in rows still exist and are visible

## Step 4: merge again using team_clean

In [124]:
merged_clean = bracket_clean.merge(
    team_lookup,
    left_on="team_clean",
    right_on="TeamName",
    how="left"
)


### What this means:

- use the cleaned names instead of the original bracket display names
- try the team match again

## Step 5: inspect the result

In [125]:
merged_clean[["region", "seed", "team", "team_clean", "is_play_in", "TeamID", "TeamName"]].head(25)


,region,seed,team,team_clean,is_play_in,TeamID,TeamName
0,East,1,Duke,Duke,False,1181.0,Duke
1,East,16,Siena,Siena,False,1373.0,Siena
2,East,8,Ole Miss,Mississippi,False,1279.0,Mississippi
3,East,9,TCU,TCU,False,1395.0,TCU
4,East,5,St. John's,St John's,False,1385.0,St John's
5,East,12,Northern Iowa,Northern Iowa,False,1320.0,Northern Iowa
6,East,4,Kansas,Kansas,False,1242.0,Kansas
7,East,13,Cal Baptist,Cal Baptist,False,1465.0,Cal Baptist
8,East,6,Louisville,Louisville,False,1257.0,Louisville
9,East,11,South Florida,South Florida,False,1378.0,South Florida


### What I'm checking:

- the names you cleaned earlier should now have TeamID
- play-in rows will probably still be unmatched
- a few tricky school names may still fail

## Step 6: see what is still unmatched

In [126]:
merged_clean[merged_clean["TeamID"].isna()][["region", "seed", "team", "team_clean", "is_play_in"]]


,region,seed,team,team_clean,is_play_in
11,East,14,North Dakota St.,North Dakota St,False
17,South,16,Lehigh / PVAMU,Lehigh / PVAMU,True
28,South,7,Saint Mary's,Saint Mary's CA,False
41,West,11,NC St. / Texas,NC St. / Texas,True
57,Midwest,11,SMU / Miami OH,SMU / Miami OH,True


- which rows are unmatched because they are play-ins
- which rows are unmatched because our name mapping is still wrong
Expected unmatched
These are okay for now because they are play-ins:

- Lehigh / PVAMU
- NC St. / Texas
- SMU / Miami OH

## Step 7: find the exact Kaggle team names for those two schools

In [127]:
m_teams[m_teams["TeamName"].str.contains("North Dakota", case=False, na=False)]


,TeamID,TeamName,FirstD1Season,LastD1Season
214,1315,North Dakota,2009,2026


In [128]:
m_teams[m_teams["TeamName"].str.contains("Mary", case=False, na=False)]


,TeamID,TeamName,FirstD1Season,LastD1Season
157,1258,Loy Marymount,1985,2026
167,1268,Maryland,1985,2026
190,1291,Mt St Mary's,1989,2026
287,1388,St Mary's CA,1985,2026
355,1456,William & Mary,1985,2026


In [129]:
m_teams[m_teams["TeamName"].str.contains("Dakota", case=False, na=False)]


,TeamID,TeamName,FirstD1Season,LastD1Season
194,1295,N Dakota St,2006,2026
214,1315,North Dakota,2009,2026
254,1355,S Dakota St,2006,2026
276,1377,South Dakota,2009,2026


## Step 8 I want to see whether Kaggle has:

N Dakota St
North Dakota
something else

In [130]:
m_teams[m_teams["TeamName"].str.contains("Dakota", case=False, na=False)]


,TeamID,TeamName,FirstD1Season,LastD1Season
194,1295,N Dakota St,2006,2026
214,1315,North Dakota,2009,2026
254,1355,S Dakota St,2006,2026
276,1377,South Dakota,2009,2026


In [131]:
name_map["Saint Mary's"] = "St Mary's CA"


## Step 9: update the mapping

In [132]:
name_map["North Dakota St."] = "N Dakota St"
name_map["Saint Mary's"] = "St Mary's CA"


## Step 10: rebuild the cleaned column and merge again

In [133]:
bracket_clean["team_clean"] = bracket_clean["team"].replace(name_map)

merged_clean = bracket_clean.merge(
    team_lookup,
    left_on="team_clean",
    right_on="TeamName",
    how="left"
)


## Step 11: check what is still unmatched

In [134]:
merged_clean[merged_clean["TeamID"].isna()][["region", "seed", "team", "team_clean", "is_play_in"]]


,region,seed,team,team_clean,is_play_in
17,South,16,Lehigh / PVAMU,Lehigh / PVAMU,True
41,West,11,NC St. / Texas,NC St. / Texas,True
57,Midwest,11,SMU / Miami OH,SMU / Miami OH,True


## Step 12: split the bracket into two groups:
- normal bracket teams
- play-in rows

In [135]:
bracket_main = merged_clean[~merged_clean["is_play_in"]].copy()
bracket_play_in = merged_clean[merged_clean["is_play_in"]].copy()


In [136]:
bracket_main.shape, bracket_play_in.shape


((61, 7), (3, 7))

## YAY: I now have bracket_main: 61 normal bracket rows and bracket_play_in 3 play-in rows!!

## Step 13: create a 2026-only feature table

In [137]:
features_2026 = team_features_named[team_features_named["Season"] == 2026].copy()


### Why: my full feature table contains many years, my bracket is for 2026, and i want to get current-season features now

## Step 14: merge the bracket teams with 2026 features

In [138]:
bracket_with_features = bracket_main.merge(
    features_2026,
    on="TeamID",
    how="left",
    suffixes=("", "_feature")
)


## Step 15: inspect whether anything is missing

In [139]:
bracket_with_features[
    ["region", "seed", "team", "TeamID", "wins", "losses", "win_pct", "avg_scoring_margin"]
].head(20)


,region,seed,team,TeamID,wins,losses,win_pct,avg_scoring_margin
0,East,1,Duke,1181.0,27.0,2.0,0.931034,20.344828
1,East,16,Siena,1373.0,20.0,11.0,0.645161,4.354839
2,East,8,Ole Miss,1279.0,12.0,17.0,0.413793,-0.482759
3,East,9,TCU,1395.0,19.0,10.0,0.655172,6.413793
4,East,5,St. John's,1385.0,23.0,6.0,0.793103,11.758621
5,East,12,Northern Iowa,1320.0,18.0,12.0,0.600000,6.266667
6,East,4,Kansas,1242.0,21.0,8.0,0.724138,7.379310
7,East,13,Cal Baptist,1465.0,21.0,8.0,0.724138,4.068966
8,East,6,Louisville,1257.0,20.0,9.0,0.689655,13.724138
9,East,11,South Florida,1378.0,20.0,8.0,0.714286,9.821429


### Checking for missing feature values

In [140]:
bracket_with_features[
    bracket_with_features["wins"].isna()
][["region", "seed", "team", "TeamID"]]


,region,seed,team,TeamID


## An empty result means:

**every non-play-in 2026 bracket team successfully matched to your 2026 feature table**

That is a big checkpoint.

I now have:

- a clean bracket table
- working team ID mapping
- play-ins isolated
- 2026 season features attached to all main bracket teams
- no missing feature rows for the main field

## Step 16: inspect the 2026 bracket table with features

In [141]:
bracket_with_features[
    ["region", "seed", "team", "wins", "losses", "win_pct", "avg_scoring_margin"]
].sort_values(["region", "seed"]).head(20)


,region,seed,team,wins,losses,win_pct,avg_scoring_margin
0,East,1,Duke,27.0,2.0,0.931034,20.344828
14,East,2,UConn,27.0,3.0,0.900000,13.533333
10,East,3,Michigan St.,24.0,5.0,0.827586,11.965517
6,East,4,Kansas,21.0,8.0,0.724138,7.379310
4,East,5,St. John's,23.0,6.0,0.793103,11.758621
8,East,6,Louisville,20.0,9.0,0.689655,13.724138
12,East,7,UCLA,19.0,10.0,0.655172,6.137931
2,East,8,Ole Miss,12.0,17.0,0.413793,-0.482759
3,East,9,TCU,19.0,10.0,0.655172,6.413793
13,East,10,UCF,20.0,8.0,0.714286,4.392857


## Step 17: build a first-round game list for one region, starting with the East

## Step 18: create a list of first-round seed pairings

In [142]:
first_round_pairs = [
    (1, 16),
    (8, 9),
    (5, 12),
    (4, 13),
    (6, 11),
    (3, 14),
    (7, 10),
    (2, 15),
]


## Step 19: filter the East region

In [143]:
east_2026 = bracket_with_features[bracket_with_features["region"] == "East"].copy()
east_2026 = east_2026.sort_values("seed")
east_2026[["seed", "team", "wins", "losses", "win_pct", "avg_scoring_margin"]]


,seed,team,wins,losses,win_pct,avg_scoring_margin
0,1,Duke,27.0,2.0,0.931034,20.344828
14,2,UConn,27.0,3.0,0.900000,13.533333
10,3,Michigan St.,24.0,5.0,0.827586,11.965517
6,4,Kansas,21.0,8.0,0.724138,7.379310
4,5,St. John's,23.0,6.0,0.793103,11.758621
8,6,Louisville,20.0,9.0,0.689655,13.724138
12,7,UCLA,19.0,10.0,0.655172,6.137931
2,8,Ole Miss,12.0,17.0,0.413793,-0.482759
3,9,TCU,19.0,10.0,0.655172,6.413793
13,10,UCF,20.0,8.0,0.714286,4.392857


## Step 20: pull one matchup manually

In [144]:
team_1 = east_2026[east_2026["seed"] == 1]
team_16 = east_2026[east_2026["seed"] == 16]

pd.concat([team_1, team_16])[["seed", "team", "wins", "losses", "win_pct", "avg_scoring_margin"]]


,seed,team,wins,losses,win_pct,avg_scoring_margin
0,1,Duke,27.0,2.0,0.931034,20.344828
1,16,Siena,20.0,11.0,0.645161,4.354839


## Step 21: build the matchup row for 1 vs 16

In [145]:
team_1_row = team_1.iloc[0]
team_16_row = team_16.iloc[0]

matchup_example = pd.DataFrame([{
    "region": "East",
    "seed_a": team_1_row["seed"],
    "team_a": team_1_row["team"],
    "seed_b": team_16_row["seed"],
    "team_b": team_16_row["team"],
    "seed_diff": team_1_row["seed"] - team_16_row["seed"],
    "win_pct_diff": team_1_row["win_pct"] - team_16_row["win_pct"],
    "scoring_margin_diff": team_1_row["avg_scoring_margin"] - team_16_row["avg_scoring_margin"],
}])

matchup_example


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,scoring_margin_diff
0,East,1,Duke,16,Siena,-15,0.285873,15.989989


## Doing one more manual matchup

In [146]:
team_8 = east_2026[east_2026["seed"] == 8]
team_9 = east_2026[east_2026["seed"] == 9]

team_8_row = team_8.iloc[0]
team_9_row = team_9.iloc[0]

matchup_8_9 = pd.DataFrame([{
    "region": "East",
    "seed_a": team_8_row["seed"],
    "team_a": team_8_row["team"],
    "seed_b": team_9_row["seed"],
    "team_b": team_9_row["team"],
    "seed_diff": team_8_row["seed"] - team_9_row["seed"],
    "win_pct_diff": team_8_row["win_pct"] - team_9_row["win_pct"],
    "scoring_margin_diff": team_8_row["avg_scoring_margin"] - team_9_row["avg_scoring_margin"],
}])

matchup_8_9


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,scoring_margin_diff
0,East,8,Ole Miss,9,TCU,-1,-0.241379,-6.896552


## Step 1: define the function shell

In [147]:
def build_first_round_matchups(region_df, region_name):
    rows = []

    for seed_a, seed_b in first_round_pairs:
        team_a_df = region_df[region_df["seed"] == seed_a]
        team_b_df = region_df[region_df["seed"] == seed_b]

        if team_a_df.empty or team_b_df.empty:
            continue

        team_a = team_a_df.iloc[0]
        team_b = team_b_df.iloc[0]

        rows.append({
            "region": region_name,
            "seed_a": team_a["seed"],
            "team_a": team_a["team"],
            "seed_b": team_b["seed"],
            "team_b": team_b["team"],
            "seed_diff": team_a["seed"] - team_b["seed"],
            "win_pct_diff": team_a["win_pct"] - team_b["win_pct"],
            "scoring_margin_diff": team_a["avg_scoring_margin"] - team_b["avg_scoring_margin"],
        })

    return pd.DataFrame(rows)




## What this means:

- region_df is something like east_2026
- region_name is "East"
- rows will collect one dictionary per game
- the loop goes through each seed pairing

# Step 2: test it on East

In [148]:
east_matchups = build_first_round_matchups(east_2026, "East")
east_matchups


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,scoring_margin_diff
0,East,1,Duke,16,Siena,-15,0.285873,15.989989
1,East,8,Ole Miss,9,TCU,-1,-0.241379,-6.896552
2,East,5,St. John's,12,Northern Iowa,-7,0.193103,5.491954
3,East,4,Kansas,13,Cal Baptist,-9,0.000000,3.310345
4,East,6,Louisville,11,South Florida,-5,-0.024631,3.902709
5,East,3,Michigan St.,14,North Dakota St.,-11,0.077586,4.322660
6,East,7,UCLA,10,UCF,-3,-0.059113,1.745074
7,East,2,UConn,15,Furman,-13,0.328571,11.247619


## A few rows stand out:

Duke vs Siena
Very strong signal for Duke.

Ole Miss vs TCU
Negative win_pct_diff and negative scoring_margin_diff, which suggests TCU may actually be stronger despite being the 9 seed.

Louisville vs South Florida
Close matchup. The differences are small.

UCLA vs UCF
Another relatively close matchup.

In [149]:
south_2026 = bracket_with_features[bracket_with_features["region"] == "South"].copy()
west_2026 = bracket_with_features[bracket_with_features["region"] == "West"].copy()
midwest_2026 = bracket_with_features[bracket_with_features["region"] == "Midwest"].copy()


In [150]:
south_2026[["seed", "team"]].sort_values("seed")


,seed,team
16,1,Florida
29,2,Houston
25,3,Illinois
21,4,Nebraska
19,5,Vanderbilt
23,6,North Carolina
27,7,Saint Mary's
17,8,Clemson
18,9,Iowa
28,10,Texas A&M


## Step 3: rerun the region builds

In [151]:
south_matchups = build_first_round_matchups(south_2026, "South")
west_matchups = build_first_round_matchups(west_2026, "West")
midwest_matchups = build_first_round_matchups(midwest_2026, "Midwest")


In [152]:
first_round_2026 = pd.concat(
    [east_matchups, south_matchups, west_matchups, midwest_matchups],
    ignore_index=True
)

first_round_2026


,region,seed_a,team_a,seed_b,team_b,seed_diff,win_pct_diff,scoring_margin_diff
0,East,1,Duke,16,Siena,-15,0.285873,15.989989
1,East,8,Ole Miss,9,TCU,-1,-0.241379,-6.896552
2,East,5,St. John's,12,Northern Iowa,-7,0.193103,5.491954
3,East,4,Kansas,13,Cal Baptist,-9,0.000000,3.310345
4,East,6,Louisville,11,South Florida,-5,-0.024631,3.902709
5,East,3,Michigan St.,14,North Dakota St.,-11,0.077586,4.322660
6,East,7,UCLA,10,UCF,-3,-0.059113,1.745074
7,East,2,UConn,15,Furman,-13,0.328571,11.247619
8,South,8,Clemson,9,Iowa,-1,0.034483,-1.862069
9,South,5,Vanderbilt,12,McNeese,-7,-0.062808,2.629310


# Building historical tournament matchup rows for training

## Step 1: start with a tiny sample

In [153]:
tourney_sample = data["tourney"].head(5).copy()
tourney_sample


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,136,1116,63,1234,54,N,0
1,1985,136,1120,59,1345,58,N,0
2,1985,136,1207,68,1250,43,N,0
3,1985,136,1229,58,1425,55,N,0
4,1985,136,1242,49,1325,38,N,0


### What I'm looking at:

- each row is one real NCAA tournament game
- WTeamID is the winner
- LTeamID is the loser
- Season tells us which year’s team features to use

## Step 2: build the smaller feature table

In [154]:
team_feature_cols = [
    "Season",
    "TeamID",
    "win_pct",
    "avg_points_for",
    "avg_points_against",
    "avg_scoring_margin",
]

team_features_small = team_features_named[team_feature_cols].copy()
team_features_small.head()


,Season,TeamID,win_pct,avg_points_for,avg_points_against,avg_scoring_margin
0,1985,1102,0.208333,63.083333,68.875000,-5.791667
1,1985,1103,0.391304,61.043478,64.086957,-3.043478
2,1985,1104,0.700000,68.500000,60.700000,7.800000
3,1985,1106,0.416667,71.625000,75.416667,-3.791667
4,1985,1108,0.760000,83.000000,75.040000,7.960000


## Step 3: attach winner features

In [155]:
winner_features = team_features_small.add_prefix("team_a_")
winner_features = winner_features.rename(
    columns={
        "team_a_Season": "Season",
        "team_a_TeamID": "WTeamID",
    }
)

sample_with_winner = tourney_sample.merge(
    winner_features,
    on=["Season", "WTeamID"],
    how="left"
)

sample_with_winner.head()


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,team_a_win_pct,team_a_avg_points_for,team_a_avg_points_against,team_a_avg_scoring_margin
0,1985,136,1116,63,1234,54,N,0,0.636364,65.333333,61.696970,3.636364
1,1985,136,1120,59,1345,58,N,0,0.620690,70.344828,66.655172,3.689655
2,1985,136,1207,68,1250,43,N,0,0.925926,75.740741,60.074074,15.666667
3,1985,136,1229,58,1425,55,N,0,0.740741,71.592593,65.629630,5.962963
4,1985,136,1242,49,1325,38,N,0,0.766667,76.033333,70.400000,5.633333


## Step 4: Prepare loser Features

In [156]:
loser_features = team_features_small.add_prefix("team_b_")
loser_features = loser_features.rename(
    columns={
        "team_b_Season": "Season",
        "team_b_TeamID": "LTeamID",
    }
)


## Step 5: merge loser features into the sample

In [157]:
sample_with_both = sample_with_winner.merge(
    loser_features,
    on=["Season", "LTeamID"],
    how="left"
)

sample_with_both.head()


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,team_a_win_pct,team_a_avg_points_for,team_a_avg_points_against,team_a_avg_scoring_margin,team_b_win_pct,team_b_avg_points_for,team_b_avg_points_against,team_b_avg_scoring_margin
0,1985,136,1116,63,1234,54,N,0,0.636364,65.333333,61.696970,3.636364,0.666667,69.733333,59.266667,10.466667
1,1985,136,1120,59,1345,58,N,0,0.620690,70.344828,66.655172,3.689655,0.680000,69.120000,65.320000,3.800000
2,1985,136,1207,68,1250,43,N,0,0.925926,75.740741,60.074074,15.666667,0.379310,65.758621,70.206897,-4.448276
3,1985,136,1229,58,1425,55,N,0,0.740741,71.592593,65.629630,5.962963,0.678571,68.392857,64.607143,3.785714
4,1985,136,1242,49,1325,38,N,0,0.766667,76.033333,70.400000,5.633333,0.740741,67.555556,63.000000,4.555556


### I now see columns like:

- team_b_win_pct
- team_b_avg_points_for
- team_b_avg_points_against
- team_b_avg_scoring_margin

## What this means: Each row is one historic NCAA tournment game, and it includes winner and loser features

## Step 6: create diff columns

In [158]:
sample_with_both["win_pct_diff"] = (
    sample_with_both["team_a_win_pct"] - sample_with_both["team_b_win_pct"]
)

sample_with_both["points_for_diff"] = (
    sample_with_both["team_a_avg_points_for"] - sample_with_both["team_b_avg_points_for"]
)

sample_with_both["points_against_diff"] = (
    sample_with_both["team_a_avg_points_against"] - sample_with_both["team_b_avg_points_against"]
)

sample_with_both["scoring_margin_diff"] = (
    sample_with_both["team_a_avg_scoring_margin"] - sample_with_both["team_b_avg_scoring_margin"]
)


### Then inspect just the useful columns

In [159]:
sample_with_both[
    [
        "Season",
        "WTeamID",
        "LTeamID",
        "team_a_win_pct",
        "team_b_win_pct",
        "win_pct_diff",
        "team_a_avg_scoring_margin",
        "team_b_avg_scoring_margin",
        "scoring_margin_diff",
    ]
].head()


,Season,WTeamID,LTeamID,team_a_win_pct,team_b_win_pct,win_pct_diff,team_a_avg_scoring_margin,team_b_avg_scoring_margin,scoring_margin_diff
0,1985,1116,1234,0.636364,0.666667,-0.030303,3.636364,10.466667,-6.830303
1,1985,1120,1345,0.620690,0.680000,-0.059310,3.689655,3.800000,-0.110345
2,1985,1207,1250,0.925926,0.379310,0.546616,15.666667,-4.448276,20.114943
3,1985,1229,1425,0.740741,0.678571,0.062169,5.962963,3.785714,2.177249
4,1985,1242,1325,0.766667,0.740741,0.025926,5.633333,4.555556,1.077778


### Right now team_a is always the winner.
So if you trained a model on this table as-is, I’d be leaking the answer into the row construction.

What comes next is fixing that setup.

The clean idea is:

- create rows where team_a and team_b are assigned in a neutral way
- add a label like team_a_won = 1 or 0

## Take one historical game row

In [160]:
game = tourney_sample.iloc[0]
game


Season     1985
DayNum      136
WTeamID    1116
WScore       63
LTeamID    1234
LScore       54
WLoc          N
NumOT         0
Name: 0, dtype: object

### Decide who is side A and side B neutrally

In [161]:
team_a_id = min(game["WTeamID"], game["LTeamID"])
team_b_id = max(game["WTeamID"], game["LTeamID"])

team_a_won = int(team_a_id == game["WTeamID"])

team_a_id, team_b_id, team_a_won


(np.int64(1116), np.int64(1234), 1)

### Get side A and side B features from the season table

In [162]:
season = game["Season"]

team_a_features = team_features_small[
    (team_features_small["Season"] == season) &
    (team_features_small["TeamID"] == team_a_id)
].iloc[0]

team_b_features = team_features_small[
    (team_features_small["Season"] == season) &
    (team_features_small["TeamID"] == team_b_id)
].iloc[0]


### Build one neutral matchup row

In [163]:
neutral_matchup_example = pd.DataFrame([{
    "Season": season,
    "team_a_id": team_a_id,
    "team_b_id": team_b_id,
    "team_a_won": team_a_won,
    "seed_diff": 0,  # placeholder for now
    "win_pct_diff": team_a_features["win_pct"] - team_b_features["win_pct"],
    "points_for_diff": team_a_features["avg_points_for"] - team_b_features["avg_points_for"],
    "points_against_diff": team_a_features["avg_points_against"] - team_b_features["avg_points_against"],
    "scoring_margin_diff": team_a_features["avg_scoring_margin"] - team_b_features["avg_scoring_margin"],
}])

neutral_matchup_example


,Season,team_a_id,team_b_id,team_a_won,seed_diff,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
0,1985,1116,1234,1,0,-0.030303,-4.4,2.430303,-6.830303


### What this row says
- team_a_id = 1116
- team_b_id = 1234
- team_a_won = 1
So side A won the game.

But the feature differences are mostly negative:

- win_pct_diff is negative
- points_for_diff is negative
- scoring_margin_diff is negative

## Take historical tournament games and turn them into a training table of neutral matchup rows

In [165]:
def build_historical_matchups(tourney_df, team_features_df):
    rows = []

    for _, game in tourney_df.iterrows():
        season = game["Season"]

        team_a_id = min(game["WTeamID"], game["LTeamID"])
        team_b_id = max(game["WTeamID"], game["LTeamID"])

        team_a_won = int(team_a_id == game["WTeamID"])

        team_a_features = team_features_df[
            (team_features_df["Season"] == season) &
            (team_features_df["TeamID"] == team_a_id)
        ]

        team_b_features = team_features_df[
            (team_features_df["Season"] == season) &
            (team_features_df["TeamID"] == team_b_id)
        ]

        if team_a_features.empty or team_b_features.empty:
            continue

        team_a_features = team_a_features.iloc[0]
        team_b_features = team_b_features.iloc[0]

        rows.append({
            "Season": season,
            "team_a_id": team_a_id,
            "team_b_id": team_b_id,
            "team_a_won": team_a_won,
            "win_pct_diff": team_a_features["win_pct"] - team_b_features["win_pct"],
            "points_for_diff": team_a_features["avg_points_for"] - team_b_features["avg_points_for"],
            "points_against_diff": team_a_features["avg_points_against"] - team_b_features["avg_points_against"],
            "scoring_margin_diff": team_a_features["avg_scoring_margin"] - team_b_features["avg_scoring_margin"],
        })

    return pd.DataFrame(rows)



### Test in on a tiny sample first

In [166]:
historical_sample_matchups = build_historical_matchups(tourney_sample, team_features_named)
historical_sample_matchups


,Season,team_a_id,team_b_id,team_a_won,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
0,1985,1116,1234,1,-0.030303,-4.400000,2.430303,-6.830303
1,1985,1120,1345,1,-0.059310,1.224828,1.335172,-0.110345
2,1985,1207,1250,1,0.546616,9.982120,-10.132822,20.114943
3,1985,1229,1425,1,0.062169,3.199735,1.022487,2.177249
4,1985,1242,1325,1,0.025926,8.477778,7.400000,1.077778


### Build the full historical matchup table

In [167]:
historical_matchups = build_historical_matchups(data["tourney"], team_features_named)
historical_matchups.head()


,Season,team_a_id,team_b_id,team_a_won,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
0,1985,1116,1234,1,-0.030303,-4.400000,2.430303,-6.830303
1,1985,1120,1345,1,-0.059310,1.224828,1.335172,-0.110345
2,1985,1207,1250,1,0.546616,9.982120,-10.132822,20.114943
3,1985,1229,1425,1,0.062169,3.199735,1.022487,2.177249
4,1985,1242,1325,1,0.025926,8.477778,7.400000,1.077778


In [168]:
historical_matchups.shape

(2585, 8)

In [169]:
historical_matchups["team_a_won"].value_counts()


team_a_won
1    1323
0    1262
Name: count, dtype: int64

### Confirmed:

how many historical tournament games turned into training rows
whether team_a_won has both classes:
- some 1
- some 0

In [170]:
historical_matchups.describe()


,Season,team_a_id,team_b_id,team_a_won,win_pct_diff,points_for_diff,points_against_diff,scoring_margin_diff
count,2585.000000,2585.000000,2585.000000,2585.000000,2585.000000,2585.000000,2585.000000,2585.000000
mean,2004.908704,1227.195745,1347.791876,0.511799,0.002907,0.908370,0.352231,0.556139
std,11.764137,82.038896,83.703438,0.499957,0.150662,8.442246,6.939126,6.872930
min,1985.000000,1101.000000,1112.000000,0.000000,-0.633333,-52.946429,-47.379464,-27.125641
25%,1995.000000,1163.000000,1277.000000,0.000000,-0.093750,-4.227586,-3.956250,-3.906250
50%,2005.000000,1227.000000,1356.000000,1.000000,0.005974,0.981061,0.267992,0.250000
75%,2015.000000,1277.000000,1423.000000,1.000000,0.101326,6.346154,4.740530,5.022059
max,2025.000000,1458.000000,1471.000000,1.000000,0.611607,45.285714,37.428571,29.057576


### What I take from it:

- all feature columns are numeric
- no missing values in the training table
- both positive and negative differences exist, which is what you want
- team_a_won is close to balanced, which is also good